# auto-j-13b inference (RQ7)

Colab (L4) record of the auto-j-13b run (`GAIR/autoj-13b-GPTQ-4bits`, vLLM's built-in GPTQ kernel, D28). `src.judge_autoj` runs clean (greedy both orders + `k_sc` sampled) and verbose (greedy both orders).

The single-prompt smoke test below only checked model load and VRAM. Its prompt is improvised, not auto-j's real template; the real template is in `src/judge_autoj.py`. Output: `runs/autoj_13b_gptq_4bits/`, copied back locally (D17).

**Step 1: Mount Google Drive where the project will be cloned**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Step 2: point to repo and pull latest changes**

In [ ]:
import getpass
!cd /content/drive/MyDrive/judge-calibration && git remote set-url origin https://github.com/senguptashubham/judge-calibration.git
token = getpass.getpass('Enter your GitHub PAT: ')
!cd /content/drive/MyDrive/judge-calibration && git pull https://{token}@github.com/senguptashubham/judge-calibration.git main

Enter your GitHub PAT: ··········
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 22 (delta 19), reused 11 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (22/22), 19.71 KiB | 1024 bytes/s, done.
From https://github.com/senguptashubham/judge-calibration
 * branch            main       -> FETCH_HEAD
Updating 5d46e2b..764f5aa
Fast-forward
 CLAUDE.md    |  2 +-
 DECISIONS.md | 34 ++++++++++++++++++++++++++++++++++
 PLAN.md      |  4 ++++
 TASKS.md     | 10 +++++++---
 4 files changed, 46 insertions(+), 4 deletions(-)


**Step 3: Install uv, create venv and set path, install dependencies**

In [ ]:
import os
%cd /content/drive/MyDrive/judge-calibration
!pip install -q uv
!uv venv /content/venv_autoj --python 3.11 --clear
!uv pip install --python /content/venv_autoj/bin/python -e ".[colab]"

os.environ["PATH"] = "/content/venv_autoj/bin:" + os.environ.get("PATH", "")


/content/drive/MyDrive/judge-calibration
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 122.7 MB/s eta 0:00:00
Using CPython 3.11.16
Creating virtual environment at: /content/venv_autoj
Activate with: source /content/venv_autoj/bin/activate
Using Python 3.11.16 environment at: /content/venv_autoj
Resolved 227 packages in 894ms
Prepared 227 packages in 33.88s
Installed 227 packages in 331ms
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==1.8.0
 + anyio==4.15.1
 + apache-tvm-ffi==0.1.11
 + arviz==0.23.4
 + astor==0.8.1
 + attrs==26.1.0
 + blake3==1.0.9
 + cachetools==7.2.0
 + cbor2==6.1.4
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + compressed-tensors==0.17.0
 + contourpy==1.3.3
 + cryptography==50.0.1
 + cuda-bindings==13.4.3
 + cuda-core==1.2.1
 + cuda-pathfinder==1.8.2
 + cuda-python==13.4.1
 + cuda-tile==1.6

**Optional: If ninja is not installed, install it**

In [ ]:
!uv pip install --python /content/venv_autoj/bin/python ninja

Using Python 3.11.16 environment at: /content/venv_autoj
Checked 1 package in 4ms


**Step 4: Single smoke test** (Optional)

In [ ]:
%%writefile /content/autoj_smoke_test.py
import time
from vllm import LLM, SamplingParams

t0 = time.time()
llm = LLM(model="GAIR/autoj-13b-GPTQ-4bits", max_model_len=4096, gpu_memory_utilization=0.85)
print(f"load time: {time.time()-t0:.1f}s")

query = "Compose an engaging travel blog post about a recent trip to Hawaii, highlighting cultural experiences and must-see attractions."
resp_a = "I recently visited Hawaii and it was wonderful. The beaches were beautiful and the food was great."
resp_b = "Hawaii is an incredible destination steeped in Polynesian culture. From the sacred hula traditions at the Polynesian Cultural Center to snorkeling in Hanauma Bay's volcanic crater, every day brought something new. It's a place where ancient tradition and modern life sit side by side."

prompt = f"""You are assessing two responses to the following query.

[Query]: {query}
[Response 1]: {resp_a}
[Response 2]: {resp_b}

Compare the two responses and give your final decision. End your answer with "So, the final decision is Response 1/Response 2/Tie"."""

out = llm.generate([prompt], SamplingParams(temperature=0.0, max_tokens=512))
print(out[0].outputs[0].text)


Overwriting /content/autoj_smoke_test.py


In [ ]:
!/content/venv_autoj/bin/python /content/autoj_smoke_test.py


INFO 09-23 06:41:40 [api_utils.py:272] non-default args: {'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'GAIR/autoj-13b-GPTQ-4bits'}
INFO 09-23 06:41:41 [model.py:672] Resolved architecture: LlamaForCausalLM
INFO 09-23 06:41:41 [model.py:2296] Downcasting torch.float32 to torch.bfloat16.
INFO 09-23 06:41:41 [model.py:1965] Using max model len 4096
INFO 09-23 06:41:44 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-23 06:41:46 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=18634) INFO 09-23 06:41:47 [core.py:122] Initializing a V1 LLM engine (v0.28.0) with config: model='GAIR/autoj-13b-GPTQ-4bits', speculative_config=None, tokenizer='GAIR/autoj-13b-GPTQ-4bits', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, m

**Step 5: check GPU use**

In [ ]:
!nvidia-smi


Thu Sep 24 02:07:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**Step 6: Smoke test on 100 items**

In [ ]:
!/content/venv_autoj/bin/python -m src.judge_autoj --config configs/run_autoj.yaml --n-items 100

INFO 09-23 08:28:50 [api_utils.py:272] non-default args: {'max_model_len': 8192, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'GAIR/autoj-13b-GPTQ-4bits'}
INFO 09-23 08:28:52 [model.py:672] Resolved architecture: LlamaForCausalLM
INFO 09-23 08:28:52 [model.py:2296] Downcasting torch.float32 to torch.bfloat16.
INFO 09-23 08:28:52 [model.py:1965] Using max model len 8192
INFO 09-23 08:28:53 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-23 08:28:55 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=10297) INFO 09-23 08:28:57 [core.py:122] Initializing a V1 LLM engine (v0.28.0) with config: model='GAIR/autoj-13b-GPTQ-4bits', speculative_config=None, tokenizer='GAIR/autoj-13b-GPTQ-4bits', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, m

**Step 7: Inference on full dataset**

In [ ]:
!/content/venv_autoj/bin/python -m src.judge_autoj --config configs/run_autoj.yaml













INFO 09-24 02:09:36 [api_utils.py:272] non-default args: {'max_model_len': 8192, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'GAIR/autoj-13b-GPTQ-4bits'}
INFO 09-24 02:09:49 [model.py:672] Resolved architecture: LlamaForCausalLM
INFO 09-24 02:09:49 [model.py:2296] Downcasting torch.float32 to torch.bfloat16.
INFO 09-24 02:09:49 [model.py:1965] Using max model len 8192
INFO 09-24 02:09:50 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-24 02:09:53 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=14180) INFO 09-24 02:09:54 [core.py:122] Initializing a V1 LLM engine (v0.28.0) with config: model='GAIR/autoj-13b-GPTQ-4bits', speculative_config=None, tokenizer='GAIR/autoj-13b-GPTQ-4bits', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch

**Step 8: Auto disconnect session after full run**

In [ ]:
from google.colab import runtime
runtime.unassign()
